# Задание 5: Матрица переходов для ДНК 1-го порядка

Строим марковскую модель 1-го порядка из реальной последовательности.

In [11]:
import numpy as np
from Bio import SeqIO

## Загрузка последовательности

In [12]:
record = next(SeqIO.parse("GCA_029856635.1_ASM2985663v1_genomic.fna", "fasta"))
seq = str(record.seq).upper()
seq = ''.join(c for c in seq if c in 'ACGT')

print(f"Длина последовательности: {len(seq)} нт")
print(f"Первые 60 нт: {seq[:60]}")

Длина последовательности: 37163 нт
Первые 60 нт: CATGAAGCTAGTTATATTGTGATTAAGCGCGGTGAGAGTATGCAGGCTACGATATAGGGC


## 1. Частоты всех 16 динуклеотидов

In [13]:
nucleotides = ['A', 'C', 'G', 'T']
nuc_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

dinuc_counts = np.zeros((4, 4), dtype=int)
for i in range(len(seq) - 1):
    dinuc_counts[nuc_index[seq[i]], nuc_index[seq[i+1]]] += 1

print("Матрица счётов динуклеотидов (строки: i, столбцы: j):")
print("     ", "  ".join(nucleotides))
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  ", dinuc_counts[i])

Матрица счётов динуклеотидов (строки: i, столбцы: j):
      A  C  G  T
A   [4382 1582 2109 3694]
C   [2134 1186 1003 2305]
G   [2306 1387 1188 1663]
T   [2946 2472 2244 4561]


## 2. Матрица переходов 4×4

$$P_{ij} = \frac{c_{ij}}{\sum_j c_{ij}}$$

In [14]:
row_sums = dinuc_counts.sum(axis=1, keepdims=True)
P = dinuc_counts / row_sums

print("Матрица переходов P:")
print("     ", "       ".join(nucleotides))
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  ", [round(x, 4) for x in P[i]])

Матрица переходов P:
      A       C       G       T
A   [np.float64(0.3724), np.float64(0.1344), np.float64(0.1792), np.float64(0.3139)]
C   [np.float64(0.322), np.float64(0.1789), np.float64(0.1513), np.float64(0.3478)]
G   [np.float64(0.3524), np.float64(0.2119), np.float64(0.1815), np.float64(0.2541)]
T   [np.float64(0.241), np.float64(0.2022), np.float64(0.1836), np.float64(0.3731)]


## 3. Проверка: сумма строк = 1

In [15]:
row_sums_check = P.sum(axis=1)
print("Суммы строк:")
for nuc, s in zip(nucleotides, row_sums_check):
    print(f"  {nuc}: {s:.6f}")


Суммы строк:
  A: 1.000000
  C: 1.000000
  G: 1.000000
  T: 1.000000


## 4. Стационарное распределение

Решаем $\pi = \pi P$ — левый собственный вектор для $\lambda = 1$.

In [16]:
eigenvalues, eigenvectors = np.linalg.eig(P.T)
idx = np.argmax(np.real(eigenvalues))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()

print("Стационарное распределение π:")
for nuc, p in zip(nucleotides, pi):
    print(f"  π({nuc}) = {p:.4f}")

Стационарное распределение π:
  π(A) = 0.3167
  π(C) = 0.1783
  π(G) = 0.1761
  π(T) = 0.3289


## 5. Сравнение со наблюдаемыми частотами

In [17]:
observed = np.array([seq.count(n) for n in nucleotides], dtype=float)
observed /= observed.sum()

print(f"{'Нукл':>6} {'π (стацион.)':>14} {'Набл. частота':>15} {'|Разность|':>12}")
print("-" * 52)
for nuc, p_stat, p_obs in zip(nucleotides, pi, observed):
    print(f"  {nuc:>4}  {p_stat:>12.4f}  {p_obs:>13.4f}  {abs(p_stat - p_obs):>12.4f}")

  Нукл   π (стацион.)   Набл. частота   |Разность|
----------------------------------------------------
     A        0.3167         0.3167        0.0000
     C        0.1783         0.1783        0.0000
     G        0.1761         0.1761        0.0000
     T        0.3289         0.3289        0.0000
